In [ ]:
#source activate newEnv
suppressMessages(library(hdf5r))
suppressMessages(library(Seurat))
suppressMessages(library(Signac))
suppressMessages(library(EnsDb.Hsapiens.v86))
suppressMessages(library(dplyr))
suppressMessages(library(ggplot2))
suppressMessages(library(Matrix))
suppressMessages(library(harmony))
suppressMessages(library(data.table))
suppressMessages(library(ggpubr))
suppressMessages(library(future))
library(sctransform)
library(reticulate)
library('Biobase')
library(gplots)
library(BiocParallel)
library(cowplot)
library(GenomeInfoDb)

In [ ]:
### Read in snATAC rds
atac_obj <- readRDS('~/data/atac_obj.rds')
### Read in snRNA rds
npod_rna <- readRDS(file= '~/data/rna_obj.rds')

# quantify gene activity
gene.activities <- GeneActivity(atac_obj)
# add gene activities as a new assay
atac_obj[["ACTIVITY_allFeatures"]] <- CreateAssayObject(counts = gene.activities)

# normalize gene activities
DefaultAssay(atac_obj) <- "ACTIVITY_allFeatures"
atac_obj <- NormalizeData(atac_obj)
atac_obj <- ScaleData(atac_obj, features = rownames(atac_obj))


In [ ]:
### Spilt object into smaller objects -- too large to run in one go
k <- 10
atac_obj$k.assign <- sample(x = 1:k, size = ncol(atac_obj), replace = TRUE)
obj.split <- SplitObject(atac_obj, split.by = 'k.assign')

# these can be written out to intermediate files to save space

In [ ]:
#### example of how to run one of the smaller objects
rm(list = ls())
npod_rna <- readRDS(file= '~/data/rna_obj.rds')
atac_curr <- readRDS(file='~/data/atac_obj_subset1.rds')


Sys.time()
# Identify anchors
transfer.anchors <- FindTransferAnchors(reference = npod_rna, query = atac_curr, features = VariableFeatures(object = npod_rna),
    reference.assay = "RNA", query.assay = "ACTIVITY_allFeatures", reduction = "cca")

celltype.predictions <- TransferData(anchorset = transfer.anchors, refdata = npod_rna$subtypes,
    weight.reduction = atac_curr[["lsi"]], dims = 2:30)

atac_curr <- AddMetaData(atac_curr, metadata = celltype.predictions)#,col.name= '2ndLT_predictedID')
options(repr.plot.width=20, repr.plot.height=10)

p1 <- DimPlot(atac_curr, group.by = "predicted.id", label = TRUE) + NoLegend() 
p1

saveRDS(atac_curr, file = '~/data/snATAC_LT_files/atac_obj_subset1_LabelTransfer.rds')
Sys.time()


In [ ]:
#### then merge of objects to get a whole label transferred object
atac1 <- readRDS('~/data/snATAC_LT_files/atac_obj_subset1_LabelTransfer.rds')
atac2 <- readRDS('~/data/snATAC_LT_files/atac_obj_subset2_LabelTransfer.rds')
atac3 <- readRDS('~/data/snATAC_LT_files/atac_obj_subset3_LabelTransfer.rds')

adata_mergeLT <- merge(atac1, y = c(atac2,atac3))


In [ ]:
## recluster
DefaultAssay(adata_mergeLT) <- 'ATAC'
adata_mergeLT <- RunTFIDF(adata_mergeLT)
adata_mergeLT <- FindTopFeatures(adata_mergeLT, min.cutoff='q0', verbose=FALSE)
adata_mergeLT <- RunSVD(adata_mergeLT)

hm_atac <- HarmonyMatrix(Embeddings(adata_mergeLT, reduction='lsi'),adata_mergeLT@meta.data,  c("library","sex","technology"), do_pca=FALSE,plot_convergence = TRUE, verbose = TRUE)
adata_mergeLT[['harmony.atac']] <- CreateDimReducObject(embeddings=hm_atac, key='atac_', assay='ATAC')
adata_mergeLT <- RunUMAP(adata_mergeLT, dims=2:30, reduction='harmony.atac', reduction.name='umap.atac', reduction.key='atacUMAP_')

adata_mergeLT <- FindNeighbors(object = adata_mergeLT, reduction = 'harmony.atac', dims = 2:30)
adata_mergeLT <- FindClusters(object = adata_mergeLT, algorithm=4,resolution = 0.5,method = "igraph") 

DimPlot(object = adata_mergeLT, label = TRUE) + NoLegend()
DimPlot(object = adata_mergeLT,group.by = "predicted.id", label = TRUE) + NoLegend()


In [ ]:
### plot some marker genes
marker.genes <- c('INS','GCG','SST','PPY','CTRB2','CFTR','PLVAP','COL1A1','C1QC','S100B',
                  'CD3D','ONECUT1','PDX1','ARX','MKI67','CDH19','CRYAB','PMP22','SCN7A',
                  'FLT4','PROX1','PDPN','LYVE1','MUC5B','TFF2','TFF3','CRISP3','CD1E',
                  'FLT3','CD1C','REG1A','REG3A','REG3G','REG1B','SPARCL1','FABP4','ITGA1','DES',
                  'COL5A2','COL6A3','LAMA2','LAMB1','SLIT2','LUM','KIT','TPSAB1','RGS13',
                  'HDC','MS4A2','HPGDS','CD69','KLRB1','CD8B','IFNG','CD19','CD79A','CD79B',
                  'NCR1','NCAM1','CD4','FOXP3')
marker.genes <- unique(marker.genes)


options(repr.plot.width=12, repr.plot.height=6)
p1 <- DotPlot(adata_mergeLT, assay='ACTIVITY_allFeatures', group.by='seurat_clusters', features=marker.genes, cluster.idents=TRUE) 
p1 <- p1 + theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('') + ylab('')
p1

options(repr.plot.width=12, repr.plot.height=6)
p1 <- DotPlot(adata_mergeLT, assay='ACTIVITY_allFeatures', group.by='predicted.id', features=marker.genes, cluster.idents=TRUE) 
p1 <- p1 + theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('') + ylab('')
p1

In [ ]:
### plot concordance of predicted labels with previous labels 
predictions <- table(adata_mergeLT$Cell_type, adata_mergeLT$predicted.id)
predictions <- predictions/rowSums(predictions)  # normalize for number of cells in each cell type
predictions <- as.data.frame(predictions)
p1 <- ggplot(predictions, aes(Var1, Var2, fill = Freq)) + geom_tile() + scale_fill_gradient(name = "Fraction of cells",
    low = "#ffffc8", high = "#7d0025") + xlab("Cell type annotation (whole snATAC object)") + ylab("Predicted cell type label (ATAC) via label transfer") +
    theme_cowplot() + theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))
p1

In [ ]:
#### since acinar subtypes did not recapitulate well in chromatin assay, we derived a cumulative acinar score 
#### for all predicted acinar cells, then retained acinar cells with a cumulative score > 0.5

adata_mergeLT$AcinarSumPrediction = rowSums(adata_mergeLT@meta.data[,c("prediction.score.Acinar_1", "prediction.score.Acinar_2",
                                                                       "prediction.score.Acinar_3","prediction.score.Acinar_4",
                                                                      "prediction.score.Acinar_5","prediction.score.Acinar_6")])
acinar <- c("Acinar_1_2_6","Acinar_3","Acinar_4","Acinar_5")

outdir2 <-'~/data/snATAC_LT_files/'
to_remove <- file.path(outdir2,"filter05_acinarSum.txt")
write.table(adata_mergeLT[[]][adata_mergeLT$FinalSubtypes %in% acinar & adata_mergeLT$AcinarSumPrediction < 0.5, c()], 
            to_remove, col.names=FALSE, quote=FALSE)

In [ ]:
to_removeFinal <-  file.path(outdir2,"filter05_acinarSum.txt")

#file.path(outdir,'harmony.lowqualclusters10.txt')
remove.cells1 <- trimws(scan(to_removeFinal, what='', sep='\n'), which='right', whitespace=' ')

remove <- c(remove.cells1)
print(length(remove))
adata_mergeLT$remove_cells <- (Cells(adata_mergeLT) %in% remove)

adata_mergeLT_filt05 <- subset(adata_mergeLT, subset=remove_cells==FALSE)
adata_mergeLT_filt05

In [ ]:
options(repr.plot.width=8, repr.plot.height=7)
DimPlot(object = adata_mergeLT_filt075,group.by = "FinalSubtypes",raster=FALSE, label = TRUE) + NoLegend()

In [ ]:
saveRDS(adata_mergeLT_filt05,'~/data/snATAC_LT_files/snATAC_Lt_filt05AcinarSum.rds')